# Nile-Chat 12B v2 — Merge, Push to Hugging Face, and Serve

Merges the v2 LoRA adapter (trained in `nile_chat_finetune_v2_colab.ipynb`) into the
base model, pushes the merged model to a **new** Hugging Face repo
(`mennaharmas/raylab-nilechat-12b-v2` — separate from the v1 repo `mennaharmas/raylab-nilechat-12b`, so the
currently-deployed v1 model is never overwritten while v2 is being validated), and
serves it with vLLM for real testing.

**Security note on the v1 notebook this is adapted from**: its serve cell had a real
Hugging Face token hardcoded in plaintext (`os.environ["HF_TOKEN"] = "hf_..."`).
It was not committed to git, but it's a live secret sitting in a local file — rotate
that token on huggingface.co/settings/tokens as a precaution, and this notebook
never hardcodes one — it reads from Colab's secret store (`userdata`) the same way
the training notebook already does for the Hub login.


### Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/gdrive')


## Stage 12 — Merge LoRA adapter into base model

In [ ]:
# Same install as the training notebook -- a fresh runtime needs it again.
!pip uninstall -y torch torchvision torchaudio vllm
!pip install -qU uv
!uv pip install --system vllm --torch-backend=auto
!pip uninstall -y torchao
!pip install -qU "transformers>=4.55.0,<=5.8.0,!=4.57.0,!=5.6.0" "datasets>=2.16.0,<=4.0.0" "accelerate>=1.3.0,<=1.15.0" "peft>=0.18.0,<=0.20.0" "trl>=0.18.0,<=0.24.0"
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
!cd LLaMA-Factory && pip install -e . --no-deps


In [ ]:
%%writefile /content/LLaMA-Factory/examples/merge_lora/raylab_merge_v2.yaml
### model
model_name_or_path: MBZUAI-Paris/Nile-Chat-12B
adapter_name_or_path: /gdrive/MyDrive/raylab-finetune-v2/models/
template: gemma
trust_remote_code: true

### export
export_dir: /content/merged_model_v2/
export_size: 5
export_device: auto  # choices: [cpu, auto]
export_legacy_format: false


In [ ]:
!cd LLaMA-Factory && llamafactory-cli export examples/merge_lora/raylab_merge_v2.yaml


### Push the merged model to Hugging Face

Same proven pattern as the v1 notebook (`notebook_login` + `create_repo` +
`upload_folder`), pointed at the new `mennaharmas/raylab-nilechat-12b-v2` repo — `private=True`, matching
the v1 precedent. Token comes from the login prompt, never hardcoded.


In [ ]:
from huggingface_hub import HfApi, notebook_login

# 1. Login -- a small box appears below this cell, paste your token (needs "Write"
#    scope) and click Login.
notebook_login()

# 2. Setup
api = HfApi()
repo_id = "mennaharmas/raylab-nilechat-12b-v2"

# 3. Create the repo (no-op if it already exists)
print("Creating repo on your account...")
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True, private=True)

# 4. Upload
print("Uploading merged model... (can take a while depending on Colab's network speed)")
api.upload_folder(
    folder_path="/content/merged_model_v2/",
    repo_id=repo_id,
    repo_type="model",
)
print(f"Model pushed to https://huggingface.co/{repo_id}")


### Step 2 — Serve the merged model on Colab GPU with vLLM

Run this in a **fresh Colab runtime** (`Runtime > Restart session`) after the merge
and push above finish — vLLM's own install brings in a different matched
`torch`/`torchvision`/`torchaudio` set than LLaMA-Factory's training environment
used.

`Runtime > Change runtime type > A100 GPU`, then confirm the GPU is attached:


In [ ]:
!nvidia-smi


If the cell above errors or shows no GPU, stop here and fix the runtime type before continuing.


In [ ]:
!pip install -q -U vllm transformers accelerate


**`torchaudio`/`torchvision`/`torch` CUDA-version fix** — same fix as the training
notebook: detect whatever torch build vLLM actually installs, reinstall
`torchvision` matched to that exact CUDA-tagged index, and leave `torchaudio`
uninstalled.


In [ ]:
import torch

torch_version = torch.__version__.split("+")[0]
torch_cuda = torch.version.cuda
cuda_tag = "cu" + torch_cuda.replace(".", "")

print(f"Detected torch=={torch_version} built for CUDA {torch_cuda} -> installing matched "
      f"torchvision from index {cuda_tag}, leaving torchaudio uninstalled")

!pip install -q "torch=={torch_version}" torchvision --index-url https://download.pytorch.org/whl/{cuda_tag}
!pip uninstall -y -q torchaudio

import subprocess
check = subprocess.run(
    ["python", "-c", "import torch, torchvision; "
     "print('torch:', torch.__version__, torch.version.cuda); "
     "print('torchvision:', torchvision.__version__)"],
    capture_output=True, text=True,
)
print(check.stdout.strip())
assert check.returncode == 0, f"torch/torchvision import failing:\n{check.stderr}"
print("torch/torchvision aligned and importable. torchaudio intentionally left uninstalled.")


In [ ]:
import os
from google.colab import userdata

# Token from Colab's secret store -- never hardcoded (see this notebook's intro cell
# re: the v1 notebook's leaked plaintext token). Only needed because the repo above
# is private.
os.environ["HF_TOKEN"] = userdata.get('huggingface')

!nohup vllm serve "mennaharmas/raylab-nilechat-12b-v2" \
    --dtype bfloat16 \
    --max-model-len 8192 \
    --gpu-memory-utilization 0.85 \
    --port 8001 \
    --served-model-name raylab-nilechat-finetuned \
    > vllm.log 2>&1 &


In [ ]:
import time

ready = False
for attempt in range(60):  # up to 10 minutes
    time.sleep(10)
    log = open("vllm.log").read() if __import__("os").path.exists("vllm.log") else ""
    if "Uvicorn running" in log or "Application startup complete" in log:
        ready = True
        break
    if "Traceback" in log and "ERROR" in log:
        print("vLLM logged an error while loading -- check the tail below.")
        break
    print(f"[{(attempt + 1) * 10}s] still loading...")

!tail -n 60 vllm.log
print("\n--- server ready:", ready, "---\n")

!curl -s http://localhost:8001/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model":"raylab-nilechat-finetuned","messages":[{"role":"user","content":"hi"}],"max_tokens":16}'


If the curl call above didn't return a real completion, stop and fix it before opening a tunnel.


In [ ]:
import os, re, time

if not os.path.exists("cloudflared-linux-amd64"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64

assert os.path.exists("cloudflared-linux-amd64") and os.path.getsize("cloudflared-linux-amd64") > 0, \
    "cloudflared download failed -- re-run this cell, or check Colab's network connectivity"

!chmod +x cloudflared-linux-amd64
!nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8001 > cloudflared.log 2>&1 &

tunnel_url = None
for _ in range(30):
    time.sleep(2)
    log = open("cloudflared.log").read() if os.path.exists("cloudflared.log") else ""
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log)
    if match:
        tunnel_url = match.group(0)
        break

assert tunnel_url, "Tunnel URL not found after 60s -- check cloudflared.log for errors and re-run this cell"
print(f"Tunnel URL: {tunnel_url}")
print("Set in your local src/.env:")
print(f"  GENERATION_BASE_URL={tunnel_url}")
print("  GENERATION_MODEL_NAME=raylab-nilechat-finetuned")


In [ ]:
!curl -s http://localhost:8001/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model":"raylab-nilechat-finetuned","messages":[{"role":"user","content":"إزيك؟"}],"max_tokens":32}'


### Quick real-case check against the live server


In [ ]:
import requests

system_prompt = """انت 'سارة'، موظفة خدمة عملاء مصرية ودودة في مركز رايلاب للأشعة والتحاليل الطبية. اتبع القواعد الآتية بالترتيب ده:

1. الدقة في استخراج الحقيقة: شغلانتك الأساسية إنك تجاوب على سؤال المريض بدقة باستخدام المعلومات المكتوبة تحت 'CONTEXT' بس. استخرج الحقيقة اللي بتجاوب على سؤاله، وبعدين اكتبها في جملة طبيعية وودودة بالعامية المصرية — من غير ما تنسخ وتلصق نص الـ CONTEXT حرفيًا زي ما هو. ممنوع تخترع سعر أو اسم أو أي حقيقة مش موجودة قدامك. لو خدمة أو حاجة معينة مكتوب في الـ CONTEXT إنها غير متاحة، قول بوضوح إنها غير متاحة (وأبدًا العكس لو مكتوب إنها متاحة). انقل الأرقام، الأسعار، والمواعيد حرفيًا كما هي مكتوبة في الـ CONTEXT لتجنب أي أخطاء حسابية أو زمنية. إذا سأل المريض عن خدمة طبية، جراحة، أو تخصص (مثل زراعة الأسنان أو الكشف الطبي) غير مذكور ومطابق حرفياً لما هو موجود في الـ CONTEXT، يجب عليك فوراً الاعتذار بلباقة وإخباره أن هذه الخدمة غير متوفرة وأن مركز رايلاب متخصص في الأشعة والتحاليل الطبية فقط. إياك أن تحاول الإجابة باستخدام معلومات عن خدمة أخرى مشابهة، وإياك أن تخترع معلومات من خارج الـ CONTEXT. ممنوع نهائيًا إنك تخترع أو تحسب أي مثال توضيحي بالأرقام من عندك، حتى لو الحساب نفسه صح رياضيًا — المريض ما طلبش الحساب ده، وهو مش مكتوب حرفيًا في الـ CONTEXT. مثال حرفي على اللي ممنوع تمامًا تعمله: لو الـ CONTEXT بيقول 'النسبه: 0.25' والمريض سأل عن نسبة الكاش باك على الأشعة، ❌ ممنوع تضيف جملة زي 'يعني مثلاً لو الأشعة تكلفتها 1000 جنيه، هترجعلك 250 جنيه' — ده مثال مُختلَق من عندك، مش موجود في الـ CONTEXT، حتى لو الحساب نفسه صح. ✅ الرد الصح هو نقل الرقم زي ما هو بس ('نسبة الكاش باك على الأشعة 25% يا فندم')، من غير أي حساب أو مثال إضافي من عندك.

2. المصطلحات الطبية: حافظ على كل المصطلحات الطبية وأسماء الفحوصات (زي MRI، CT، X-Ray، CBC) والأسماء التجارية بالإنجليزي بالظبط زي ما هي مكتوبة في الـ CONTEXT. أي كلمة إنجليزي عامة مش مصطلح طبي (زي 'Services' أو 'Branches') ترجمها للعربي (ممنوع نهائيًا تعريب أو ترجمة أسماء الأشعة والفحوصات، يجب نقلها بالإنجليزي دائمًا كما هي في الـ CONTEXT، حتى لو كان باقي الرد بالعربي).

3. الشخصية والأسلوب: اتكلمي بعامية مصرية طبيعية وصافية 100% — من غير فصحى رسمية جامدة، ومن غير أي لهجة خليجية (ممنوع تمامًا استخدام كلمات خليجية مثل: وش، شلون، أبغى، وايد، أو الفصحى المعقدة). تحدثي بأسلوب الشارع المصري الراقي والودود.

4. أمثلة على الأسلوب المطلوب (جمل كاملة طبيعية، مش كلمات منفصلة لازم تتكرر حرفيًا):
   - الـ CONTEXT بيقول: 'الجمعة مغلق'. المريض: 'مواعيد الجمعة؟' ← الرد: 'يوم الجمعة الفرع بيكون إجازة يا فندم، تحب أحجزلك في يوم تاني؟'
   - الـ CONTEXT بيقول: 'اسانسير: متاح'. المريض: 'فيه أسانسير؟' ← الرد: 'أيوه فيه أسانسير في الفرع يا فندم، تحب تعرف حاجة تانية؟'
   - الـ CONTEXT بيقول: 'فيزا: متاح. فاليو: غير متاح'. المريض: 'بتقبلوا فيزا؟' ← الرد: 'أيوه، الفرع بيقبل فيزا عادي، بس للأسف الفاليو مش متاحة حاليًا. حابب تعرف طريقة دفع تانية؟'

5. سؤال المتابعة: ادمج سؤال المتابعة في نهاية الرد كجملة طبيعية متصلة، وممنوع كتابة أي عناوين وصفية قبله.

6. الأسئلة العامة والواسعة: لو المريض سأل سؤال عام عن الخدمات المتاحة بشكل عام (زي 'عندكم إيه من الأشعة')، اقرأ كل الـ CONTEXT (هيوصلك مقسّم لمصادر مرقمة [BEGIN SOURCE n]...[END SOURCE n]) وطلّع قائمة نقطية بسيطة وواضحة بالعربي للخدمات المتاحة، وخلي كل حقيقة مرتبطة بمصدرها الصح. خليها مختصرة جدًا.

كمان، لو سؤال المريض عن حقيقة محددة (زي مدة تحضير، حد أقصى للوزن، مدة زمنية، أو أي رقم أو شرط معين) — مش سؤال عام عن قائمة خدمات — لكن وصلك أكتر من [BEGIN SOURCE] في نفس الرد:
   - لو أكتر من مصدر بيقول نفس الحقيقة بالظبط (نفس الرقم أو نفس الشرط) لكن كل مصدر مرتبط بفرع مختلف، والمريض ما حددش أي فرع — قول الحقيقة عادي وبثقة من غير ما تسأل عن الفرع أصلاً، لأن الإجابة واحدة في كل الحالات.
   - لو مصدر بيتكلم عن فحص أو خدمة مختلفة تمامًا عن اللي المريض سأل عنها — حتى لو شكله أو تنسيقه (زي جدول أو تصنيف بالأرقام) قريب من اللي محتاجه — تجاهل المصدر ده تمامًا وما تستخدمش أرقامه أو شروطه. حدد المصدر الصح بناءً على إن موضوعه يطابق بالظبط الفحص أو الخدمة اللي المريض سأل عنها، مش مجرد شكل البيانات أو تنسيقها.
   - ممنوع نهائيًا إنك تردي برسالة فاضية أو تكرري سؤال المريض من غير إجابة لمجرد إن قدامك أكتر من مصدر أو قيم متعارضة. لو الحقيقة الصح موجودة في مصدر واحد على الأقل بيتكلم عن نفس اللي اتسأل عنه بالظبط، لازم تقوليها بثقة.
"""

def generate_via_vllm(system, instruction, input_text):
    url = "http://localhost:8001/v1/chat/completions"
    headers = {"Content-Type": "application/json"}
    payload = {
        "model": "raylab-nilechat-finetuned",
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": f"{instruction}\n{input_text}"},
        ],
        "temperature": 0.0,
        "max_tokens": 400,
    }
    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]

print(generate_via_vllm(
    system_prompt,
    "CONTEXT:\n[Document: Branch Directory] معلومات الفرع الإسعاف: متاح",
    "PATIENT MESSAGE:\nلو حصل طارئ وأنا في فرع حلوان، فيه عربية إسعاف موجودة؟",
))


### Full val.json evaluation against the live server

Exact-JSON-match scoring (strict — real wording variance in the phrasing doesn't
count against it, only whether the extracted JSON matches byte-for-byte). Points at
the v2 dataset's `val.json` (15 examples).


In [ ]:
import json
import requests
import re

val_file_path = "/gdrive/MyDrive/raylab-finetune-v2/datasets/val.json"

with open(val_file_path, "r", encoding="utf-8") as f:
    val_data = json.load(f)

total_examples = len(val_data)
correct_answers = 0

print(f"\nStarting evaluation on {total_examples} examples using vLLM Server...\n")
print("-" * 50)

url = "http://localhost:8001/v1/chat/completions"
headers = {"Content-Type": "application/json"}

for i, example in enumerate(val_data):
    system_prompt = example.get("system", "")
    instruction = example.get("instruction", "")
    input_text = example.get("input", "")
    expected_output = example.get("output", "").strip()

    payload = {
        "model": "raylab-nilechat-finetuned",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"{instruction}\n{input_text}".strip()},
        ],
        "temperature": 0.0,
        "max_tokens": 400,
    }

    try:
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()
        generated_text = response.json()["choices"][0]["message"]["content"].strip()
    except Exception as e:
        print(f"Error connecting to vLLM: {e}")
        continue

    is_correct = False
    expected_match = re.search(r'```json(.*?)```', expected_output, re.DOTALL)
    expected_json_str = expected_match.group(1).strip() if expected_match else "{}"
    generated_match = re.search(r'```json(.*?)```', generated_text, re.DOTALL)
    generated_json_str = generated_match.group(1).strip() if generated_match else "{}"

    try:
        expected_json = json.loads(expected_json_str)
        generated_json = json.loads(generated_json_str)
        if expected_json == generated_json:
            is_correct = True
    except json.JSONDecodeError:
        pass

    if is_correct:
        correct_answers += 1

    print(f"Example {i+1}/{total_examples}")
    print(f"Correct: {'PASS' if is_correct else 'FAIL'}")
    if not is_correct:
        print(f"Expected JSON: {expected_json_str}")
        print(f"Generated JSON: {generated_json_str}")
    print("-" * 30)

accuracy_percentage = (correct_answers / total_examples) * 100
print("\n" + "=" * 50)
print(f"Total Validation Examples: {total_examples}")
print(f"Correct Answers: {correct_answers}")
print(f"Final Accuracy: {accuracy_percentage:.2f}%")
print("=" * 50)
